# Phase 4: Data Cleaning

## NexaCart Marketplace Intelligence

### Objective
This notebook performs data cleaning based on the findings from the Data Assessment & Validation phase.

Cleaning activities include:

- Removing invalid records
- Removing duplicate records where necessary
- Standardizing text fields
- Preparing cleaned datasets for feature engineering
- Exporting cleaned datasets

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
PROJECT_ROOT = Path("../")

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw"
CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "cleaned"

CLEAN_DATA_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
excel_file = RAW_DATA_PATH / "NexaCart_Data.xlsx"

xls = pd.ExcelFile(excel_file)

print("Available Sheets:")
print(xls.sheet_names)

Available Sheets:
['sellers_dataset', 'product_category_name_translati', 'products_dataset', 'order_reviews_dataset', 'orders_dataset', 'order_payments_dataset', 'order_items_dataset', 'geolocation_dataset', 'customers_dataset']


In [5]:
sellers = pd.read_excel(
    excel_file,
    sheet_name="sellers_dataset"
)

translation = pd.read_excel(
    excel_file,
    sheet_name="product_category_name_translati"
)

products = pd.read_excel(
    excel_file,
    sheet_name="products_dataset"
)

reviews = pd.read_excel(
    excel_file,
    sheet_name="order_reviews_dataset"
)

orders = pd.read_excel(
    excel_file,
    sheet_name="orders_dataset"
)

payments = pd.read_excel(
    excel_file,
    sheet_name="order_payments_dataset"
)

order_items = pd.read_excel(
    excel_file,
    sheet_name="order_items_dataset"
)

geolocation = pd.read_excel(
    excel_file,
    sheet_name="geolocation_dataset"
)

customers = pd.read_excel(
    excel_file,
    sheet_name="customers_dataset"
)

In [6]:
datasets = {
    "Sellers": sellers,
    "Translation": translation,
    "Products": products,
    "Reviews": reviews,
    "Orders": orders,
    "Payments": payments,
    "Order Items": order_items,
    "Geolocation": geolocation,
    "Customers": customers
}

for name, df in datasets.items():
    print(f"{name:<20} {df.shape}")

Sellers              (3095, 4)
Translation          (72, 2)
Products             (32951, 9)
Reviews              (99224, 5)
Orders               (99441, 8)
Payments             (103886, 5)
Order Items          (112650, 7)
Geolocation          (1000163, 5)
Customers            (99441, 5)


In [7]:
sellers_clean = sellers.copy()

translation_clean = translation.copy()

products_clean = products.copy()

reviews_clean = reviews.copy()

orders_clean = orders.copy()

payments_clean = payments.copy()

order_items_clean = order_items.copy()

geolocation_clean = geolocation.copy()

customers_clean = customers.copy()

In [8]:
cleaning_log = pd.DataFrame(
    columns=[
        "Dataset",
        "Cleaning Step",
        "Records Affected"
    ]
)

cleaning_log

,Dataset,Cleaning Step,Records Affected


In [9]:
translation_clean.columns

Index(['Column1', 'Column2'], dtype='str')

In [10]:
translation_clean.columns = [
    "product_category_name",
    "product_category_name_english"
]

translation_clean.head()

,product_category_name,product_category_name_english
0,product_category_name,product_category_name_english
1,beleza_saude,health_beauty
2,informatica_acessorios,computers_accessories
3,automotivo,auto
4,cama_mesa_banho,bed_bath_table


In [11]:
cleaning_log.loc[len(cleaning_log)] = [
    "Translation",
    "Renamed columns",
    2
]

cleaning_log

,Dataset,Cleaning Step,Records Affected
0,Translation,Renamed columns,2


In [12]:
print("Before Cleaning :", geolocation_clean.shape)

Before Cleaning : (1000163, 5)


In [13]:
before = len(geolocation_clean)

geolocation_clean = geolocation_clean.drop_duplicates()

removed_duplicates = before - len(geolocation_clean)

print("Duplicate Rows Removed :", removed_duplicates)
print("Current Shape :", geolocation_clean.shape)

Duplicate Rows Removed : 261831
Current Shape : (738332, 5)


In [14]:
before = len(geolocation_clean)

geolocation_clean = geolocation_clean[
    geolocation_clean["geolocation_lat"].between(-34, 6)
]

removed_lat = before - len(geolocation_clean)

print("Invalid Latitude Records Removed :", removed_lat)

Invalid Latitude Records Removed : 27


In [15]:
before = len(geolocation_clean)

geolocation_clean = geolocation_clean[
    geolocation_clean["geolocation_lng"].between(-75, -30)
]

removed_lng = before - len(geolocation_clean)

print("Invalid Longitude Records Removed :", removed_lng)

Invalid Longitude Records Removed : 0


In [16]:
geolocation_clean["geolocation_city"] = (
    geolocation_clean["geolocation_city"]
        .str.lower()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
)

In [17]:
geolocation_clean.reset_index(drop=True, inplace=True)

In [18]:
print("After Cleaning :", geolocation_clean.shape)

After Cleaning : (738305, 5)


In [19]:
cleaning_log.loc[len(cleaning_log)] = [
    "Geolocation",
    "Removed duplicate rows",
    removed_duplicates
]

cleaning_log.loc[len(cleaning_log)] = [
    "Geolocation",
    "Removed invalid latitude records",
    removed_lat
]

cleaning_log.loc[len(cleaning_log)] = [
    "Geolocation",
    "Removed invalid longitude records",
    removed_lng
]

cleaning_log.loc[len(cleaning_log)] = [
    "Geolocation",
    "Standardized city names",
    len(geolocation_clean)
]

cleaning_log

,Dataset,Cleaning Step,Records Affected
0,Translation,Renamed columns,2
1,Geolocation,Removed duplicate rows,261831
2,Geolocation,Removed invalid latitude records,27
3,Geolocation,Removed invalid longitude records,0
4,Geolocation,Standardized city names,738305


In [20]:
translation_clean = translation_clean.iloc[1:].reset_index(drop=True)

translation_clean.head()

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [21]:
cleaning_log.loc[len(cleaning_log)] = [
    "Translation",
    "Removed duplicated header row",
    1
]

In [22]:
print("Translation Shape:", translation_clean.shape)

translation_clean.head()

Translation Shape: (71, 2)


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


In [23]:
print("Shape:", geolocation_clean.shape)

print("Duplicate Rows:", geolocation_clean.duplicated().sum())

print(
    "Invalid Latitude:",
    (~geolocation_clean["geolocation_lat"].between(-34, 6)).sum()
)

print(
    "Invalid Longitude:",
    (~geolocation_clean["geolocation_lng"].between(-75, -30)).sum()
)

print(
    "Missing Values:",
    geolocation_clean.isnull().sum().sum()
)

Shape: (738305, 5)
Duplicate Rows: 0
Invalid Latitude: 0
Invalid Longitude: 0
Missing Values: 0


In [24]:
cleaning_summary = pd.DataFrame({
    "Dataset": [
        "Translation",
        "Geolocation"
    ],
    "Records Before": [
        len(translation),
        len(geolocation)
    ],
    "Records After": [
        len(translation_clean),
        len(geolocation_clean)
    ]
})

cleaning_summary["Rows Removed"] = (
    cleaning_summary["Records Before"]
    - cleaning_summary["Records After"]
)

cleaning_summary

,Dataset,Records Before,Records After,Rows Removed
0,Translation,72,71,1
1,Geolocation,1000163,738305,261858


In [25]:
from pathlib import Path

# Create output directory
output_dir = Path("../data/cleaned")
output_dir.mkdir(parents=True, exist_ok=True)

# Export datasets
sellers.to_csv(output_dir / "sellers.csv", index=False)

translation_clean.to_csv(
    output_dir / "product_category_translation.csv",
    index=False
)

products.to_csv(output_dir / "products.csv", index=False)

reviews.to_csv(output_dir / "reviews.csv", index=False)

orders.to_csv(output_dir / "orders.csv", index=False)

payments.to_csv(output_dir / "payments.csv", index=False)

order_items.to_csv(output_dir / "order_items.csv", index=False)

customers.to_csv(output_dir / "customers.csv", index=False)

geolocation_clean.to_csv(
    output_dir / "geolocation.csv",
    index=False
)

print("All cleaned datasets exported successfully.")

All cleaned datasets exported successfully.


In [26]:
report_dir = Path("../reports")
report_dir.mkdir(parents=True, exist_ok=True)

cleaning_log.to_csv(
    report_dir / "cleaning_log.csv",
    index=False
)

cleaning_summary.to_csv(
    report_dir / "cleaning_summary.csv",
    index=False
)

print("Cleaning reports exported successfully.")

Cleaning reports exported successfully.


In [27]:
with pd.ExcelWriter(output_dir / "NexaCart_Cleaned.xlsx") as writer:

    sellers.to_excel(writer, sheet_name="sellers", index=False)
    translation_clean.to_excel(writer, sheet_name="translation", index=False)
    products.to_excel(writer, sheet_name="products", index=False)
    reviews.to_excel(writer, sheet_name="reviews", index=False)
    orders.to_excel(writer, sheet_name="orders", index=False)
    payments.to_excel(writer, sheet_name="payments", index=False)
    order_items.to_excel(writer, sheet_name="order_items", index=False)
    customers.to_excel(writer, sheet_name="customers", index=False)
    geolocation_clean.to_excel(writer, sheet_name="geolocation", index=False)

print("Excel workbook exported.")

Excel workbook exported.
